# REHABELT (PREDICTION MODEL)

###### FORCE INSTALL PYTHON 3.9

In [ ]:
%%bash
# 1. Download Miniconda (Python 3.9 Installer)
MINICONDA_INSTALLER_SCRIPT=Miniconda3-py39_4.12.0-Linux-x86_64.sh
MINICONDA_PREFIX=/usr/local
wget https://repo.anaconda.com/miniconda/$MINICONDA_INSTALLER_SCRIPT

# 2. Install it over the current Python
chmod +x $MINICONDA_INSTALLER_SCRIPT
./$MINICONDA_INSTALLER_SCRIPT -b -f -p $MINICONDA_PREFIX

# 3. Delete the installer to save space
rm $MINICONDA_INSTALLER_SCRIPT

PREFIX=/usr/local
Unpacking payload ...
Solving environment: ...working... done

## Package Plan ##

  environment location: /usr/local

  added / updated specs:
    - _libgcc_mutex==0.1=main
    - _openmp_mutex==4.5=1_gnu
    - brotlipy==0.7.0=py39h27cfd23_1003
    - ca-certificates==2022.3.29=h06a4308_1
    - certifi==2021.10.8=py39h06a4308_2
    - cffi==1.15.0=py39hd667e15_1
    - charset-normalizer==2.0.4=pyhd3eb1b0_0
    - colorama==0.4.4=pyhd3eb1b0_0
    - conda-content-trust==0.1.1=pyhd3eb1b0_0
    - conda-package-handling==1.8.1=py39h7f8727e_0
    - conda==4.12.0=py39h06a4308_0
    - cryptography==36.0.0=py39h9ce1e76_0
    - idna==3.3=pyhd3eb1b0_0
    - ld_impl_linux-64==2.35.1=h7274673_9
    - libffi==3.3=he6710b0_2
    - libgcc-ng==9.3.0=h5101ec6_17
    - libgomp==9.3.0=h5101ec6_17
    - libstdcxx-ng==9.3.0=hd4cf53a_17
    - ncurses==6.3=h7f8727e_2
    - openssl==1.1.1n=h7f8727e_0
    - pip==21.2.4=py39h06a4308_0
    - pycosat==0.6.3=py39h27cfd23_0
    - pycparser==2.21=pyhd3

--2026-01-24 11:17:11--  https://repo.anaconda.com/miniconda/Miniconda3-py39_4.12.0-Linux-x86_64.sh
Resolving repo.anaconda.com (repo.anaconda.com)... 104.16.191.158, 104.16.32.241, 2606:4700::6810:bf9e, ...
Connecting to repo.anaconda.com (repo.anaconda.com)|104.16.191.158|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 76607678 (73M) [application/x-sh]
Saving to: ‘Miniconda3-py39_4.12.0-Linux-x86_64.sh’

     0K .......... .......... .......... .......... ..........  0% 48.4M 2s
    50K .......... .......... .......... .......... ..........  0% 3.71M 11s
   100K .......... .......... .......... .......... ..........  0%  203M 7s
   150K .......... .......... .......... .......... ..........  0%  210M 5s
   200K .......... .......... .......... .......... ..........  0% 6.90M 6s
   250K .......... .......... .......... .......... ..........  0%  197M 5s
   300K .......... .......... .......... .......... ..........  0%  257M 5s
   350K .......... .......... .

##### DOWNGRADE TENSORFLOW (Fixes "Version 9" Crash)

In [ ]:
!pip uninstall -y tensorflow
!pip install tensorflow==2.10.0

Found existing installation: tensorflow 2.10.0
Uninstalling tensorflow-2.10.0:
  Successfully uninstalled tensorflow-2.10.0
  Using cached tensorflow-2.10.0-cp39-cp39-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (578.1 MB)


###### MOUNT GOOGLE DRIVE

In [1]:
from google.colab import drive
import os

drive.mount('/content/drive', force_remount=True)

# Verify the main folder exists
folder_path = '/content/drive/MyDrive/rehabelt_data'
if os.path.exists(folder_path):
    print(f"\nSUCCESS: Found main folder: {folder_path}")
else:
    print(f"\nERROR: Main folder '{folder_path}' not found.")

Mounted at /content/drive

SUCCESS: Found main folder: /content/drive/MyDrive/rehabelt_data


##### DATA AUGMENTATION SCRIPT

In [6]:
%%bash
pip install -q pandas openpyxl scipy numpy

cat > augment_all.py <<EOF
import os
import glob
import numpy as np
import pandas as pd
from scipy.interpolate import interp1d

# PATH CONFIGURATION
RAW_BASE_DIR = '/content/drive/MyDrive/rehabelt_data'

OUTPUT_FOLDER = '/content/drive/MyDrive/Commission_Files/AUGMENTED_MASTER'

os.makedirs(OUTPUT_FOLDER, exist_ok=True)

DIRS = {
    'WALKING':   os.path.join(RAW_BASE_DIR, 'raw_walk'),
    'SIT_STAND': os.path.join(RAW_BASE_DIR, 'raw_sit_stand'),
    'SUPINE':    os.path.join(RAW_BASE_DIR, 'raw_supine_sit')
}

if not os.path.exists(OUTPUT_FOLDER): os.makedirs(OUTPUT_FOLDER)

# AUGMENTATION FUNCTIONS
def add_noise(data, noise_level=0.5):
    noise = np.random.normal(0, noise_level, data.shape)
    return data + noise

def scale_data(data, scale_factor):
    return data * scale_factor

def time_warp(data, rate):
    x_old = np.linspace(0, 1, len(data))
    x_new = np.linspace(0, 1, int(len(data) * rate))
    f = interp1d(x_old, data, axis=0, kind='linear', fill_value="extrapolate")
    return f(x_new)

# PROCESSING LOOP
total_files = 0

if not os.path.exists(RAW_BASE_DIR):
    print(f"ERROR: Input folder '{RAW_BASE_DIR}' not found!")
    print("Please create 'Client_Raw' in your Google Drive and put the 3 subfolders inside.")
    exit()

for activity_name, folder_path in DIRS.items():
    if not os.path.exists(folder_path):
        print(f"WARNING: Subfolder '{folder_path}' not found. Skipping...")
        continue

    # Look for both .xlsx and .XLSX
    files = glob.glob(os.path.join(folder_path, "*.[xX][lL][sS][xX]"))
    print(f"Processing {activity_name}: Found {len(files)} files...")

    for file in files:
        try:
            df = pd.read_excel(file)
            raw_data = df.iloc[:, 0:6].dropna().values

            if len(raw_data) < 50:
                print(f"  Skipping {os.path.basename(file)}: Too short")
                continue

            base_name = f"{activity_name}_{os.path.basename(file).replace('.xlsx', '')}"

            # ORIGINAL
            pd.DataFrame(raw_data).to_csv(os.path.join(OUTPUT_FOLDER, f"{base_name}_ORIG.csv"), index=False, header=['ax','ay','az','gx','gy','gz'])
            total_files += 1

            # VARIATIONS
            for i in range(5):
                aug = add_noise(raw_data, noise_level=np.random.uniform(0.05, 0.5))
                pd.DataFrame(aug).to_csv(os.path.join(OUTPUT_FOLDER, f"{base_name}_NOISE_{i}.csv"), index=False, header=['ax','ay','az','gx','gy','gz'])
                total_files += 1

            for i in range(5):
                aug = scale_data(raw_data, scale_factor=np.random.uniform(0.85, 1.15))
                pd.DataFrame(aug).to_csv(os.path.join(OUTPUT_FOLDER, f"{base_name}_SCALE_{i}.csv"), index=False, header=['ax','ay','az','gx','gy','gz'])
                total_files += 1

            for i in range(5):
                aug = time_warp(raw_data, rate=np.random.uniform(0.8, 1.2))
                pd.DataFrame(aug).to_csv(os.path.join(OUTPUT_FOLDER, f"{base_name}_SPEED_{i}.csv"), index=False, header=['ax','ay','az','gx','gy','gz'])
                total_files += 1

            for i in range(5):
                aug = time_warp(add_noise(raw_data, 0.2), rate=np.random.uniform(0.9, 1.1))
                pd.DataFrame(aug).to_csv(os.path.join(OUTPUT_FOLDER, f"{base_name}_COMBO_{i}.csv"), index=False, header=['ax','ay','az','gx','gy','gz'])
                total_files += 1

        except Exception as e:
            print(f"  ERROR in {file}: {e}")

print(f"\n DONE! Created {total_files} training files in {OUTPUT_FOLDER}")
EOF

python augment_all.py

Processing WALKING: Found 45 files...
Processing SIT_STAND: Found 45 files...
Processing SUPINE: Found 40 files...

 DONE! Created 2730 training files in /content/drive/MyDrive/Commission_Files/AUGMENTED_MASTER


##### MAIN SCRIPT

In [7]:
%%bash

# SETUPS
rm -rf legacy_env Miniconda3-py39_4.12.0-Linux-x86_64.sh
wget -q https://repo.anaconda.com/miniconda/Miniconda3-py39_4.12.0-Linux-x86_64.sh
chmod +x Miniconda3-py39_4.12.0-Linux-x86_64.sh
./Miniconda3-py39_4.12.0-Linux-x86_64.sh -b -f -p ./legacy_env > /dev/null
./legacy_env/bin/python -m pip install -q tensorflow==2.10.0 pandas "numpy<2" scikit-learn matplotlib seaborn scipy

# MAIN SCRIPT
cat > builder.py <<EOF
import os
import glob
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical
from sklearn.utils import class_weight
from sklearn.preprocessing import StandardScaler
from scipy.signal import resample
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

# CONFIGURATION
INPUT_FOLDER = '/content/drive/MyDrive/Commission_Files/AUGMENTED_MASTER'
OUTPUT_FOLDER = '/content/drive/MyDrive/Commission_Files'
TARGET_WINDOW_SIZE = 100

# HYPERPARAMETERS
LEARNING_RATE = 0.0001
EPOCHS = 100
BATCH_SIZE = 128

if not os.path.exists(OUTPUT_FOLDER): os.makedirs(OUTPUT_FOLDER)

# LOAD DATA
csv_files = glob.glob(os.path.join(INPUT_FOLDER, "**/*.csv"), recursive=True)
X_all, y_all = [], []

print(f"Processing {len(csv_files)} files from AUGMENTED_MASTER...")

for file in csv_files:
    try:
        df = pd.read_csv(file)
        data = df.iloc[:, 0:6].values
        name = os.path.basename(file).upper()

        label = -1
        if "SUPINE" in name: label = 2
        elif "SIT_STAND" in name: label = 1
        elif "WALKING" in name: label = 0

        if label == -1: continue

        if len(data) != TARGET_WINDOW_SIZE:
             data = resample(data, TARGET_WINDOW_SIZE)

        if label == 0 or np.sum(np.std(data, axis=0)) > 0.5:
             X_all.append(data)
             y_all.append(label)
    except: pass

X = np.array(X_all)
y = np.array(y_all)
print(f"Loaded {len(X)} windows.")

# NORMALIZE
scaler = StandardScaler()
X_flat = X.reshape(-1, 6)
X_scaled = scaler.fit_transform(X_flat).reshape(X.shape)

# UNDEFINED CLASS (Class 3)
n_noise = int(len(X) / 3)
X_noise = []
for i in range(n_noise):
    X_noise.append(np.random.normal(0, 1, (100, 6)))

X_final = np.concatenate((X_scaled, np.array(X_noise)), axis=0)
y_final = np.concatenate((y, np.full(n_noise, 3)), axis=0)

# FLATTEN FOR DENSE NETWORK
X_final = X_final.reshape(X_final.shape[0], -1)

y_integers = y_final.astype(int)
class_weights = class_weight.compute_class_weight('balanced', classes=np.unique(y_integers), y=y_integers)
cw_dict = dict(enumerate(class_weights))

y_encoded = to_categorical(y_final, 4)
X_train, X_val, y_train, y_val = train_test_split(X_final, y_encoded, test_size=0.2, stratify=y_encoded)

# DENSE MODEL
model = Sequential([
    Dense(128, activation='relu', input_shape=(600,)),
    Dropout(0.5),
    Dense(64, activation='relu'),
    Dropout(0.5),
    Dense(4, activation='softmax')
])

# COMPILE
opt = Adam(learning_rate=LEARNING_RATE)
model.compile(optimizer=opt, loss='categorical_crossentropy', metrics=['accuracy'])

print(f"Starting Training: LR={LEARNING_RATE}, Epochs={EPOCHS}, Batch={BATCH_SIZE}")
model.fit(X_train, y_train, epochs=EPOCHS, batch_size=BATCH_SIZE, validation_data=(X_val, y_val), verbose=1, class_weight=cw_dict)

# EVALUATION REPORT
y_pred_probs = model.predict(X_val)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = np.argmax(y_val, axis=1)
class_names = ['Walking', 'Sit-Stand', 'Supine-Sit', 'Unrecognized']

print("DETAILED PERFORMANCE REPORT")
print(classification_report(y_true, y_pred, target_names=class_names))

# SAVE CONFUSION MATRIX
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix')
save_path = os.path.join(OUTPUT_FOLDER, 'confusion_matrix.png')
plt.savefig(save_path, bbox_inches='tight')
print(f"Confusion Matrix saved to: {save_path}")

# SAVE MODEL HEADER
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS]
tflite_model = converter.convert()

def hex_to_c_array(data):
    c_str = '#include <cstdint>\\n'
    c_str += f'const unsigned int model_data_len = {len(data)};\\n'
    c_str += f'alignas(16) const unsigned char model_data[] = {{\\n  '
    hex_array = [f'0x{val:02x}' for val in data]
    for i in range(0, len(hex_array), 12):
        c_str += ', '.join(hex_array[i:i+12])
        if i + 12 < len(hex_array): c_str += ",\\n  "
    c_str += '\\n};\\n'
    return c_str

with open(os.path.join(OUTPUT_FOLDER, 'model_data.h'), "w") as f:
    f.write(hex_to_c_array(tflite_model))

print(f"\\n DONE!")
print(f"MEAN: {scaler.mean_}")
print(f"SCALE: {scaler.scale_}")
EOF

# RUN
MPLBACKEND=Agg ./legacy_env/bin/python builder.py

Processing 2730 files from AUGMENTED_MASTER...
Loaded 2708 windows.
Starting Training: LR=0.0001, Epochs=100, Batch=128
Epoch 1/100
23/23 [==============================] - 1s 25ms/step - loss: 1.9968 - accuracy: 0.3137 - val_loss: 0.9995 - val_accuracy: 0.5997
Epoch 2/100
23/23 [==============================] - 0s 10ms/step - loss: 1.4954 - accuracy: 0.4273 - val_loss: 0.7867 - val_accuracy: 0.7258
Epoch 3/100
23/23 [==============================] - 0s 11ms/step - loss: 1.2705 - accuracy: 0.5093 - val_loss: 0.6780 - val_accuracy: 0.7950
Epoch 4/100
23/23 [==============================] - 0s 9ms/step - loss: 1.1127 - accuracy: 0.5637 - val_loss: 0.6036 - val_accuracy: 0.8449
Epoch 5/100
23/23 [==============================] - 0s 13ms/step - loss: 0.9825 - accuracy: 0.6035 - val_loss: 0.5477 - val_accuracy: 0.8560
Epoch 6/100
23/23 [==============================] - 0s 8ms/step - loss: 0.9447 - accuracy: 0.6212 - val_loss: 0.4996 - val_accuracy: 0.8740
Epoch 7/100
23/23 [===========

2026-02-07 09:30:05.374244: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-02-07 09:30:05.541675: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory
2026-02-07 09:30:05.541710: I tensorflow/stream_executor/cuda/cudart_stub.cc:29] Ignore above cudart dlerror if you do not have a GPU set up on your machine.
2026-02-07 09:30:05.578046: E tensorflow/stream_executor/cuda/cuda_blas.cc:2981] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-02-07 09:30:06.518089: W tensorflow/stream_executor/platform/de